# GPU-Accelerated Batched MLP Inference — Colab runner

Runs the CPU/CUDA-naive/CUDA-tiled benchmark pipeline on a Colab GPU runtime.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator = GPU (T4 is fine).


## 1. Check the GPU and toolchain

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version

## 2. Get the code

Set `REPO_URL` and `BRANCH` if you're not using the default.

In [ ]:
REPO_URL = "https://github.com/Aivon99/ArchitecturesAndPlatformsForAI.git"
BRANCH = "main"
REPO_DIR = "/content/ArchitecturesAndPlatformsForAI"

import os
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

## 3. Configure and build (Release)

In [ ]:
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j

## 4. Correctness check

Cross-checks CPU / GPU-naive / GPU-tiled outputs against each other before trusting any timing numbers.

In [ ]:
!ctest --test-dir build --output-on-failure

## 5. Run the benchmark sweep

Writes `results/benchmark.csv` and `results/device_info.csv` 

Flags : `--depth`, `--repeats`, `--warmup`, `--max-batch`, `--kernel {cpu_naive,gpu_naive,gpu_tiled}`, `--out`.

In [ ]:
!./build/benchmark --out results/benchmark.csv

## 6. Preview results

In [ ]:
import pandas as pd

df = pd.read_csv("results/benchmark.csv")
display(df)

device_info = pd.read_csv("results/device_info.csv", header=None, names=["key", "value"])
display(device_info)

## 7. Save results


In [ ]:
from google.colab import files

files.download("results/benchmark.csv")
files.download("results/device_info.csv")

In [ ]:
# persist to Google Drive instead of downloading each time.
# from google.colab import drive
# drive.mount('/content/drive')
# DEST = "/content/drive/MyDrive/mlp_inference_results"
# import shutil, os
# os.makedirs(DEST, exist_ok=True)
# shutil.copy("results/benchmark.csv", DEST)
# shutil.copy("results/device_info.csv", DEST)